In [0]:
import os
import time
from datetime import datetime
import gc
import json
from dbruntime.databricks_repl_context import get_context
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from datetime import datetime, timedelta

In [0]:
# ── Configuration ────────────────────────────────────────────────────────
BOOTSTRAP_SERVERS = "pkc-619z3.us-east1.gcp.confluent.cloud:9092"
TOPIC_NAME            = "xml_topic"   
# ── Confluent Cloud credentials (SASL_SSL) ───────────────────────────────
#   Replace these with your Confluent Cloud API key & secret, or use dbutils.secrets.get()
API_KEY    = "ZATXMTIJZNGATCSM"
API_SECRET = "cfltBHf2fbnYXbaxEVai4Ia2L3P1nczO1ATF1PFs16OoVGqo/qeaIdsAXBkoKqdA"

jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'


In [0]:
# --- Top Cell: Environment Detection ---

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

prod_workspace_urls = {
    "arpi-otp-prod-we-adb-01": "adb-7405610440211744.4.azuredatabricks.net"
}

dev_workspace_urls = {
    "arpi-otp-dev-we-adb-01": "dbc-c65cc629-3bd6.cloud.databricks.com"
}

psq_workspace_urls = {
    "arpi-otp-psq-we-adb-01": "adb-7405612720040592.12.azuredatabricks.net"
}

if workspace_url in psq_workspace_urls.values():
    _env = "psq"
elif workspace_url in prod_workspace_urls.values():
    _env = "prod"
elif workspace_url in dev_workspace_urls.values():
    _env = "dev"
else:
    raise ValueError(
        f"Workspace URL '{workspace_url}' not recognised. "
        f"Register it in prod_workspace_urls, psq_workspace_urls, or dev_workspace_urls."
    )




In [0]:

# ################################################## #
# Catalog & Framework Table References              #
# All notebooks use these variables - no hardcoded catalog names #
# ################################################## #

catalog_map = {"dev": "streaming_dev", "psq": "uc_otp_psq", "prod": "uc_otp_prod"}
catalog_name = catalog_map[_env]

# Friendly environment label for logging
_env_label_map = {"dev": "DEV", "psq": "PSQ", "prod": "PROD"}
_env_label = _env_label_map.get(_env, _env)

# Control table (streaming specific metadata)
STREAMING_METADATA_TABLE = f"{catalog_name}.f_control.streaming_ingestion_metadata"

# Shared audit table (same table used by batch framework)
AUDIT_TABLE = f"{catalog_name}.f_control.generic_job_audit"

print(f"Environment: {_env_label}")
print(f"Catalog: {catalog_name}")
print(f"Streaming Metadata Table: {STREAMING_METADATA_TABLE}")
print(f"Audit Table: {AUDIT_TABLE}")

In [0]:
catalog_name=f"streaming_{_env}"
print("catalog_name: ",catalog_name)

In [0]:
STREAMING_METADATA_TABLE=f"streaming_{_env}.f_control.streaming_ingestion_metadata"

In [0]:
from datetime import datetime

# ==============================================================================
# EH_ENV_CONFIG
# ==============================================================================

EH_ENV_CONFIG = {
    "_dev": {
        "read_namespace": "grp-otp-dev-ehub-02",
        "write_namespace": "grp-otp-dev-ehub-01",
        "service_credential": "sv_otp_dev_eventhub",
    },
    "_psq": {
        "read_namespace": "grp-otp-psq-ehub-02",
        "write_namespace": "grp-otp-psq-ehub-01",
        "service_credential": "sv_otp_psq_eventhub",
    },
    "_prod": {
        "read_namespace": "grp-otp-prd-ehub-02",
        "write_namespace": "grp-otp-prd-ehub-01",
        "service_credential": "sv_otp_prd_eventhub",
    },
}

# ------------------------------------------------------------------------------
# Source Event Hub Registry (INBOUND - CSNX -> Databricks)
# DEV:  grp-otp-dev-ehub-02  | Resource: grp-otp-dev-ac-01
# PSQ:  grp-otp-psq-ehub-02  | Resource: grp-otp-psq-ac-01
# PROD: grp-otp-prd-ehub-02  | Resource: grp-otp-prd-ac-01
# Partitions: 2 | Consumer group: eip-databricks-fnr-{name}-{env}
# Note: PROD uses -prd suffix (not -prod)
# ------------------------------------------------------------------------------

SOURCE_EVENT_HUBS = {
    "_dev": [
        {"event_hub_name": "ehub-csnx-invadjustments", "consumer_group": "eip-databricks-fnr-invadjustments-dev"},
        {"event_hub_name": "ehub-csnx-plannedreceipts", "consumer_group": "eip-databricks-fnr-plannedreceipts-dev"},
        {"event_hub_name": "ehub-csnx-planshipments", "consumer_group": "eip-databricks-fnr-planshipments-dev"},
        {"event_hub_name": "ehub-csnx-invupdates", "consumer_group": "eip-databricks-fnr-invupdates-dev"},
        {"event_hub_name": "ehub-csnx-invupdatesdatedproducts", "consumer_group": "eip-databricks-fnr-invupdtdatedprod-dev"},
    ],
    "_psq": [
        {"event_hub_name": "ehub-csnx-invadjustments", "consumer_group": "eip-databricks-fnr-invadjustments-psq"},
        {"event_hub_name": "ehub-csnx-plannedreceipts", "consumer_group": "eip-databricks-fnr-plannedreceipts-psq"},
        {"event_hub_name": "ehub-csnx-planshipments", "consumer_group": "eip-databricks-fnr-planshipments-psq"},
        {"event_hub_name": "ehub-csnx-invupdates", "consumer_group": "eip-databricks-fnr-invupdates-psq"},
        {"event_hub_name": "ehub-csnx-invupdatesdatedproducts", "consumer_group": "eip-databricks-fnr-invupdtdatedprod-psq"},
    ],
    "_prod": [
        {"event_hub_name": "ehub-csnx-invadjustments", "consumer_group": "eip-databricks-fnr-invadjustments-prd"},
        {"event_hub_name": "ehub-csnx-plannedreceipts", "consumer_group": "eip-databricks-fnr-plannedreceipts-prd"},
        {"event_hub_name": "ehub-csnx-planshipments", "consumer_group": "eip-databricks-fnr-planshipments-prd"},
        {"event_hub_name": "ehub-csnx-invupdates", "consumer_group": "eip-databricks-fnr-invupdates-prd"},
        {"event_hub_name": "ehub-csnx-invupdatesdatedproducts", "consumer_group": "eip-databricks-fnr-invupdtdatedprod-prd"},
    ],
}

SOURCE_EVENT_HUBS = SOURCE_EVENT_HUBS.get(_env, [])
EH_PARTITIONS = 2

print(f"\n Source Event Hubs ({len(SOURCE_EVENT_HUBS)}):")
for _eh in SOURCE_EVENT_HUBS:
    print(f"    {_eh['event_hub_name']:40s} {_eh['consumer_group']}")

if _env in EH_ENV_CONFIG:
    _eh_cfg = EH_ENV_CONFIG[_env]
    EH_NAMESPACE = _eh_cfg["read_namespace"]
    EH_BOOTSTRAP_SERVERS = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
    EH_WRITE_NAMESPACE = _eh_cfg["write_namespace"]
    EH_WRITE_BOOTSTRAP_SERVERS = f"{EH_WRITE_NAMESPACE}.servicebus.windows.net:9093"
    EH_SERVICE_CREDENTIAL = _eh_cfg["service_credential"]

    print(f"  Read Namespace       : {EH_NAMESPACE}")
    print(f"  Read Bootstrap       : {EH_BOOTSTRAP_SERVERS}")
    print(f"  Write Namespace      : {EH_WRITE_NAMESPACE}")
    print(f"  Write Bootstrap      : {EH_WRITE_BOOTSTRAP_SERVERS}")
    print(f"  Service Credential   : {EH_SERVICE_CREDENTIAL}")
else:
    print(f"  \u26a0 [{_env_label}] Event Hub not configured for this environment")
    EH_NAMESPACE = None
    EH_BOOTSTRAP_SERVERS = None
    EH_WRITE_NAMESPACE = None
    EH_WRITE_BOOTSTRAP_SERVERS = None
    EH_SERVICE_CREDENTIAL = None


# ==============================================================================
# Outbound Event Hub Registry (all 10 confirmed Event Hubs)
# Consumer groups are for downstream consumers (warehouse),
# not used by Databricks (we are the publisher/sender).
# CG pattern: grp-eip-whse-{env}-cg-{name}-01
# Note: PROD uses -prd suffix (not -prod)
# ==============================================================================

OUTBOUND_EVENT_HUBS = {
    "_dev": [
        {"event_hub_name": "ehub-fnr-adjustments", "consumer_group": "grp-eip-whse-dev-cg-adjustments-01"},
        {"event_hub_name": "ehub-fnr-spoilages", "consumer_group": "grp-eip-whse-dev-cg-spoilages-01"},
        {"event_hub_name": "ehub-fnr-inventorychecks", "consumer_group": "grp-eip-whse-dev-cg-inventorychecks-01"},
        {"event_hub_name": "ehub-fnr-deliveries", "consumer_group": "grp-eip-whse-dev-cg-deliveries-01"},
        {"event_hub_name": "ehub-fnr-openpo", "consumer_group": "grp-eip-whse-dev-cg-openpo-01"},
        {"event_hub_name": "ehub-fnr-openso", "consumer_group": "grp-eip-whse-dev-cg-openso-01"},
        {"event_hub_name": "ehub-fnr-sales", "consumer_group": "grp-eip-whse-dev-cg-sales-01"},
        {"event_hub_name": "ehub-fnr-balances", "consumer_group": "grp-eip-whse-dev-cg-balances-01"},
        {"event_hub_name": "ehub-fnr-batchbalances", "consumer_group": "grp-eip-whse-dev-cg-batchbalances-01"},
        {"event_hub_name": "ehub-fnr-transfers", "consumer_group": "grp-eip-whse-dev-cg-transfers-01"},
    ],
    "_psq": [
        {"event_hub_name": "ehub-fnr-adjustments", "consumer_group": "grp-eip-whse-psq-cg-adjustments-01"},
        {"event_hub_name": "ehub-fnr-spoilages", "consumer_group": "grp-eip-whse-psq-cg-spoilages-01"},
        {"event_hub_name": "ehub-fnr-inventorychecks", "consumer_group": "grp-eip-whse-psq-cg-inventorychecks-01"},
        {"event_hub_name": "ehub-fnr-deliveries", "consumer_group": "grp-eip-whse-psq-cg-deliveries-01"},
        {"event_hub_name": "ehub-fnr-openpo", "consumer_group": "grp-eip-whse-psq-cg-openpo-01"},
        {"event_hub_name": "ehub-fnr-openso", "consumer_group": "grp-eip-whse-psq-cg-openso-01"},
        {"event_hub_name": "ehub-fnr-sales", "consumer_group": "grp-eip-whse-psq-cg-sales-01"},
        {"event_hub_name": "ehub-fnr-balances", "consumer_group": "grp-eip-whse-psq-cg-balances-01"},
        {"event_hub_name": "ehub-fnr-batchbalances", "consumer_group": "grp-eip-whse-psq-cg-batchbalances-01"},
        {"event_hub_name": "ehub-fnr-transfers", "consumer_group": "grp-eip-whse-psq-cg-transfers-01"},
    ],
    "_prod": [
        {"event_hub_name": "ehub-fnr-adjustments", "consumer_group": "grp-eip-whse-prd-cg-adjustments-01"},
        {"event_hub_name": "ehub-fnr-spoilages", "consumer_group": "grp-eip-whse-prd-cg-spoilages-01"},
        {"event_hub_name": "ehub-fnr-inventorychecks", "consumer_group": "grp-eip-whse-prd-cg-inventorychecks-01"},
        {"event_hub_name": "ehub-fnr-deliveries", "consumer_group": "grp-eip-whse-prd-cg-deliveries-01"},
        {"event_hub_name": "ehub-fnr-openpo", "consumer_group": "grp-eip-whse-prd-cg-openpo-01"},
        {"event_hub_name": "ehub-fnr-openso", "consumer_group": "grp-eip-whse-prd-cg-openso-01"},
        {"event_hub_name": "ehub-fnr-sales", "consumer_group": "grp-eip-whse-prd-cg-sales-01"},
        {"event_hub_name": "ehub-fnr-balances", "consumer_group": "grp-eip-whse-prd-cg-balances-01"},
        {"event_hub_name": "ehub-fnr-batchbalances", "consumer_group": "grp-eip-whse-prd-cg-batchbalances-01"},
        {"event_hub_name": "ehub-fnr-transfers", "consumer_group": "grp-eip-whse-prd-cg-transfers-01"},
    ],
}

OUTBOUND_EVENT_HUBS = OUTBOUND_EVENT_HUBS.get(_env, [])

print(f"\n Outbound Event Hubs ({len(OUTBOUND_EVENT_HUBS)}):")
for _eh in OUTBOUND_EVENT_HUBS:
    print(f"    {_eh['event_hub_name']:40s} {_eh['consumer_group']}")


# ------------------------------------------------------------------------------
# Per-BCK Gold Event Hub Publish Configs
# Mapping: source pipeline (BCK) -> outbound Event Hub(s)
# A single BCK can publish to multiple outbound Event Hubs.
# Developers: update query + event_hub_name for your pipeline.
# ------------------------------------------------------------------------------
# BCK -> Outbound Event Hub Fan-Out Mapping (confirmed Sept 2026)
# 5 source BCKs -> 10 outbound Event Hubs
# Gold custom notebooks join silver data with batch master data
# to produce each output. Each entry has its own gold table/query.
#
# is_active=False for new entries -> flip to True once the gold
# custom notebook and gold table are ready.
# Remap event_hub_name if the actual output changes.
# ------------------------------------------------------------------------------

# FNR_INVADJUSTMENTS -> ehub-fnr-adjustments + ehub-fnr-spoilages
FNR_INVADJUSTMENTS_EVENT_HUB = [
    {
        "query": f"SELECT * FROM {catalog_name}.g_csnx.gold_invadjustments",
        "event_hub_name": "ehub-fnr-adjustments",
        "kafka_key": "FNR_INVADJUSTMENTS",
        "is_active": True,
    },
    {
        "query": f"SELECT * FROM {catalog_name}.g_csnx.gold_spoilages",
        "event_hub_name": "ehub-fnr-spoilages",
        "kafka_key": "FNR_SPOILAGES",
        "is_active": False,  # Activate when gold_spoilages table exists
    },
]

# FNR_PLANNEDRECEIPTS -> ehub-fnr-deliveries + ehub-fnr-openpo
FNR_PLANNEDRECEIPTS_EVENT_HUB = [
    # Continued in remaining script modules...
]


# ==============================================================================
# Kafka Option Builders for Azure Event Hub
# Used by STREAMING_INGESTION (read) and GOLD_EVENTHUB (write)
# ==============================================================================

def build_eh_kafka_options(
    event_hub_name,
    consumer_group="$Default",
    starting_offsets="earliest",
    max_offsets_per_trigger=None,
    fail_on_data_loss=False,
    include_ending_offsets=False,
):
    """
    Build Kafka options dict for reading from Azure Event Hub.

    Parameters
    ----------
    event_hub_name : str
        The Event Hub (topic) name to subscribe to.
    consumer_group : str
        Kafka consumer group (default: '$Default').
    starting_offsets : str
        'earliest', 'latest', or a JSON offsets string.
    max_offsets_per_trigger : int, optional
        Cap on events per micro-batch (streaming only).
    fail_on_data_loss : bool
        Whether to fail if offsets are missing (default: False).
    include_ending_offsets : bool
        Set True for batch reads (adds endingOffsets='latest').

    Returns
    -------
    dict
        Spark readStream options for .format('kafka').
    """
    opts = {
        "kafka.bootstrap.servers": EH_BOOTSTRAP_SERVERS,
        "subscribe": event_hub_name,
        "startingOffsets": starting_offsets,
        "kafka.group.id": consumer_group,
        "failOnDataLoss": str(fail_on_data_loss).lower(),
        "kafka.request.timeout.ms": "60000",
        "kafka.session.timeout.ms": "30000",
    }

    # Auth: UC Service Credential (primary) or SAS connection string (Key Vault fallback)
    if EH_SERVICE_CREDENTIAL:
        opts["databricks.serviceCredential"] = EH_SERVICE_CREDENTIAL
    elif EH_CONNECTION_STRING:
        _sasl_jaas = (
            'org.apache.kafka.common.security.plain.PlainLoginModule required '
            'f\'username="$ConnectionString" '
            f'f\'password="{EH_CONNECTION_STRING}";\''
        )
        opts.update({
            "kafka.security.protocol": "SASL_SSL",
            "kafka.sasl.mechanism": "PLAIN",
            "kafka.sasl.jaas.config": _sasl_jaas,
        })

    if max_offsets_per_trigger is not None:
        opts["maxOffsetsPerTrigger"] = str(max_offsets_per_trigger)
    if include_ending_offsets:
        opts["endingOffsets"] = "latest"

    return opts


def build_eh_kafka_write_options():
    """
    Build Kafka options dict for writing to Azure Event Hub.

    The topic must be set separately via .option('topic', <event_hub_name>).

    Returns
    -------
    dict
        Spark write options for .format('kafka').
    """
    opts = {
        "kafka.bootstrap.servers": EH_WRITE_BOOTSTRAP_SERVERS,
    }

    # Auth: UC Service Credential only for write path
    if EH_SERVICE_CREDENTIAL:
        opts["databricks.serviceCredential"] = EH_SERVICE_CREDENTIAL

    return opts


print("\u2705 build_eh_kafka_options() + build_eh_kafka_write_options() loaded")